In [1]:
import mysql.connector
from contextlib import contextmanager

In [2]:
@contextmanager
def get_db_cursor(commit=False):
    connection=mysql.connector.connect(
        host='localhost',
        user='root',
        password='root',
        database='expense_manager'
    )
#    if connection.is_connected():
#        print("Connected")
#    else:
#        print("Not Connected")

    #cursor=connection.cursor()                             # gives results in tuple format
    cursor=connection.cursor(dictionary=True)             # gives results in dictionary format
    yield cursor
    #return connection, cursor
    if commit:
        connection.commit()
    #print("closing cursor")
    cursor.close()
    connection.close()

In [3]:
def fetch_all_records():
    with get_db_cursor() as cursor:
        cursor.execute('SELECT * FROM expenses')
        expenses=cursor.fetchall()
        for expense in expenses:
            print(expense)

In [14]:
def fetch_expenses_for_date(expense_date):
    with get_db_cursor() as cursor:
        cursor.execute('SELECT * FROM expenses WHERE expense_date=%s',(expense_date,))
        expenses=cursor.fetchall()
        return expenses

In [15]:
def insert_expense(expense_date,amount,category,notes):
    with get_db_cursor(commit=True) as cursor:
        cursor.execute('INSERT into expenses (expense_date,amount,category,notes) VALUES (%s,%s,%s,%s)',(expense_date,amount,category,notes))

        #with get_db_cursor(commit=True) as cursor:
        #cursor.execute("INSERT INTO expenses (expense_date,amount,category,notes) VALUES (%s, %s, %s, %s)",(expense_date, amount, category, notes))

In [16]:
def delete_expenses_for_date(expense_date):
    with get_db_cursor(commit=True) as cursor:
        cursor.execute('DELETE FROM expenses WHERE expense_date=%s',(expense_date,))

In [17]:
def fetch_expense_summary(start_date,end_date):
    with get_db_cursor() as cursor:
        cursor.execute('SELECT category, SUM(amount) as total FROM expenses WHERE expense_date BETWEEN %s and %s GROUP by category',(start_date,end_date))
        expenses=cursor.fetchall()
        return expenses

In [18]:
if __name__=='__main__':
    #expenses=fetch_expenses_for_date("2024-08-01")
    #print(expenses)
    expenses=fetch_expense_summary('2024-08-01','2024-08-05')
    for expense in expenses:
        print(expense)

{'category': 'Entertainment', 'total': 225.0}
{'category': 'Shopping', 'total': 670.0}
{'category': 'Food', 'total': 2335.0}
{'category': 'Other', 'total': 90.0}
{'category': 'Rent', 'total': 2777.0}
